#### Visão Geral
##### Schema : gold
##### Table : fact_pedidos_itens

| Detalhe | Informação |
|---------|------------|
| Criado Originalmente Por | Wellikiandre Bosich |
| Tabela de Dados de Saída | `{environment}.gold.fact_pedidos_itens` |
| Origem Fonte de Dados de Entrada | Camada silver |
| Destino Fonte de Dados de Saída | Camada gold |

#### Histórico

| Data       | Desenvolvido Por         | Motivo                                         |
|:----------:|--------------------------|-----------------------------------------------|
| 04/06/2026 | Wellikiandre Bosich    | Criação da fato transacional de itens de pedidos na Gold. |

In [ ]:
%run ../0_Config/0-Init

In [ ]:
# Parâmetros de Inicialização
table_name = 'fact_pedidos_itens'
output_path_data = f"{var_gold}/{table_name}/data"
table_name_schema = f'{var_environment}.{var_gold_schema}.{table_name}'

In [ ]:
from pyspark.sql.functions import col, sha2, date_format, coalesce, lit, when
from pyspark.sql.types import DecimalType

df_cab = spark.read.table(f"{var_environment}.{var_silver_schema}.case_erp_pedidos_cabecalho")
df_item = spark.read.table(f"{var_environment}.{var_silver_schema}.case_erp_pedidos_itens")

df_cli = spark.read.table(f"{var_environment}.{var_gold_schema}.dim_clientes")
df_prod = spark.read.table(f"{var_environment}.{var_gold_schema}.dim_produtos")
df_vend = spark.read.table(f"{var_environment}.{var_gold_schema}.dim_vendedores")

# Junção dos pedidos
df_pedidos = df_item.join(df_cab, "id_pedido", "inner")

# Cruzamento com dimensões para obter as SKs
df_fact = (
    df_pedidos
    .join(df_cli, "id_cliente", "left")
    .join(df_prod, "id_produto", "left")
    .join(df_vend, "id_vendedor", "left")
    
    .withColumn("sk_tempo", date_format(col("data_pedido"), "yyyyMMdd").cast("integer"))
    .withColumn("sk_cliente", coalesce(col("sk_cliente"), sha2(lit("-1"), 256)))
    .withColumn("sk_produto", coalesce(col("sk_produto"), sha2(lit("-1"), 256)))
    .withColumn("sk_vendedor", coalesce(col("sk_vendedor"), sha2(lit("-1"), 256)))
    
    .withColumn("valor_bruto", (col("quantidade") * col("preco_unitario")).cast(DecimalType(12, 2)))
    .withColumn("valor_liquido", when(col("status_pedido") == "Cancelado", lit(0.00)).otherwise(col("valor_bruto")).cast(DecimalType(12, 2)))
    
    .select(
        col("id_item_pedido").alias("id_fato_item_pedido"),
        col("id_pedido").cast("integer").alias("id_pedido"),
        col("sk_cliente"),
        col("sk_produto"),
        col("sk_vendedor"),
        col("sk_tempo"),
        col("quantidade"),
        col("preco_unitario"),
        col("valor_bruto"),
        col("valor_liquido"),
        col("status_pedido").alias("status_pedido_evento")
    )
)

In [ ]:
df_write_fact = df_fact.withColumn("data_key_str", col("sk_tempo").cast("string"))

import pyspark.sql.functions as F

process_fact(
    df_write=df_write_fact,
    nome_gravacao_tabela=table_name_schema,
    caminho_gravacao_tabela=output_path_data,
    data_formatada="data_key_str",
    chave_clusterby=["sk_tempo"]
)